In [ ]:
!git clone https://github.com/JesperLybeck/Visual-Saliency.git
%cd /content/Visual-Saliency

/content
Cloning into 'Visual-Saliency'...
remote: Enumerating objects: 2122, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 2122 (delta 0), reused 1 (delta 0), pack-reused 2119 (from 2)
Receiving objects: 100% (2122/2122), 174.51 MiB | 61.75 MiB/s, done.
Resolving deltas: 100% (56/56), done.
Updating files: 100% (4034/4034), done.
/content/Visual-Saliency


In [2]:
import kagglehub
from pathlib import Path
import os

project_dir = Path.cwd().resolve()
dataset_dir = project_dir / "dataset"
download_dir = dataset_dir / "salicon"
path = kagglehub.dataset_download("roshan401/salicon", output_dir=str(download_dir), force_download=True)
print("Downloaded to:", path)

100%|██████████| 3.97G/3.97G [00:29<00:00, 147MB/s]

Extracting files...


Downloaded to: /content/Visual-Saliency/dataset/salicon


In [ ]:
from resNet_trans_pretrain_256 import SaliencyResNetTransPretrained
from dataLoader import get_dataloaders

#DATASET_ROOT = Path("/kaggle/input/salicon")
DATASET_ROOT = Path("/content/Visual-Saliency/dataset/salicon")

train_loader, val_loader, test_loader = get_dataloaders(
    batch_size=28,
    dataset_root=DATASET_ROOT
)
image, heatmap, fixation_map = next(iter(train_loader))

Getting dataloaders...
Image dir: /content/Visual-Saliency/dataset/salicon/images/images/train
Image dir exists: True
Heatmap dir: /content/Visual-Saliency/dataset/salicon/maps/train
Heatmap dir exists: True
Matched 10000 pairs

Image dir: /content/Visual-Saliency/dataset/salicon/images/images/val
Image dir exists: True
Heatmap dir: /content/Visual-Saliency/dataset/salicon/maps/val
Heatmap dir exists: True
Matched 5000 pairs



In [ ]:
import torch
import torch.nn.functional as F
from tqdm import tqdm
from typing import Optional, Any, Dict
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

def _to_device(batch, device):
    x, y = batch
    return x.to(device), y.to(device)

def kl_spatial_loss(preds: torch.Tensor, targets: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    B = preds.shape[0]
    pred_flat = preds.view(B, -1)
    tgt_flat = targets.view(B, -1).float()

    tgt_sum = tgt_flat.sum(dim=1, keepdim=True)
    tgt_prob = tgt_flat / (tgt_sum + eps)

    zero_mask = (tgt_sum.squeeze(1) == 0)
    if zero_mask.any():
        tgt_prob[zero_mask] = 1.0 / tgt_prob.shape[1]

    pred_log = F.log_softmax(pred_flat, dim=1)
    return F.kl_div(pred_log, tgt_prob, reduction="batchmean")

def batch_cc(preds: torch.Tensor, targets: torch.Tensor, eps: float = 1e-8) -> float:
    if preds.ndim == 4 and preds.shape[1] == 1:
        p = preds.view(preds.shape[0], -1)
    else:
        p = preds.view(preds.shape[0], -1)
    t = targets.view(targets.shape[0], -1).float()

    p_mean = p.mean(dim=1, keepdim=True)
    t_mean = t.mean(dim=1, keepdim=True)
    p_z = p - p_mean
    t_z = t - t_mean

    num = (p_z * t_z).sum(dim=1)
    den = torch.sqrt((p_z ** 2).sum(dim=1) * (t_z ** 2).sum(dim=1) + eps)
    cc = num / den
    return float(cc.mean().item())

def cc_loss(preds, targets, eps=1e-8):
    p = preds.view(preds.shape[0], -1)
    t = targets.view(targets.shape[0], -1).float()

    p = p - p.mean(dim=1, keepdim=True)
    t = t - t.mean(dim=1, keepdim=True)

    cc = (p * t).sum(dim=1) / (
        torch.sqrt((p ** 2).sum(dim=1) * (t ** 2).sum(dim=1)) + eps
    )
    return cc.mean()


def nss_loss(pred, fixation, eps=1e-8):
    pred = (pred - pred.mean(dim=[2, 3], keepdim=True)) / (
        pred.std(dim=[2, 3], keepdim=True) + eps
    )
    nss = (pred * fixation).sum(dim=[2, 3]) / (
        fixation.sum(dim=[2, 3]) + eps
    )
    return -nss.mean()

def combined_saliency_loss(preds, saliency_map, fixation_map):
    pred_pos = F.softplus(preds)
    kl = kl_spatial_loss(preds, saliency_map)
    cc = cc_loss(pred_pos, saliency_map)
    nss = -nss_loss(pred_pos, fixation_map)

    return kl + 0.5 * (1 - cc) - 0.3 * nss

def save_checkpoint(path, model, optimizer, epoch, best_val, history):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val": best_val,
        "history": history
    }, path)

def load_checkpoint(path, model, optimizer, device):
    checkpoint = torch.load(path, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    return (
        checkpoint["epoch"],
        checkpoint["best_val"],
        checkpoint["history"]
    )

def train_one_epoch(model: torch.nn.Module, dataloader: torch.utils.data.DataLoader, optimizer: torch.optim.Optimizer, device: str, clip_grad: Optional[float] = None) -> Dict[str, float]:
    model.train()
    running_loss = 0.0
    running_kl = 0.0
    running_cc = 0.0
    running_nss = 0.0
    n = 0
    pbar = tqdm(dataloader, desc="Train", leave=False)

    for batch in pbar:
        x, saliency, fixation = batch
        x = x.to(device)
        saliency = saliency.to(device)
        fixation = fixation.to(device)

        optimizer.zero_grad()
        preds = model(x)
        pred_pos = F.softplus(preds.detach())

        kl = kl_spatial_loss(preds, saliency)
        loss = combined_saliency_loss(preds, saliency, fixation)
        loss.backward()
        if clip_grad is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
        optimizer.step()

        bs = x.shape[0]
        running_loss += loss.item() * bs
        running_kl += kl.item() * bs
        running_cc += batch_cc(pred_pos, saliency.detach()) * bs
        running_nss += (-nss_loss(pred_pos, fixation.detach()).item()) * bs
        n += bs
        pbar.set_postfix(loss=loss.item())

    return {"loss": running_loss / n, "kl": running_kl / n, "cc": running_cc / n, "nss": running_nss / n}

def validate(model: torch.nn.Module, dataloader: torch.utils.data.DataLoader, device: str) -> Dict[str, float]:
    model.eval()
    running_loss = 0.0
    running_kl = 0.0
    running_cc = 0.0
    running_nss = 0.0
    n = 0
    pbar = tqdm(dataloader, desc="Valid", leave=False)
    with torch.no_grad():
        for batch in pbar:
            x, saliency, fixation = batch
            x = x.to(device)
            saliency = saliency.to(device)
            fixation = fixation.to(device)

            preds = model(x)
            kl = kl_spatial_loss(preds, saliency)
            loss = combined_saliency_loss(preds, saliency, fixation)
            pred_pos = F.softplus(preds.detach())

            bs = x.shape[0]
            running_loss += loss.item() * bs
            running_kl += kl.item() * bs
            running_cc += batch_cc(pred_pos, saliency) * bs
            running_nss += (-nss_loss(pred_pos, fixation).item()) * bs
            n += bs
            pbar.set_postfix(loss=loss.item())

    return {"loss": running_loss / n, "kl": running_kl / n, "cc": running_cc / n, "nss": running_nss / n}

def fit(model, train_loader, valid_loader, optimizer, device,
        epochs=10,
        save_path="best.pt",
        checkpoint_path="checkpoint.pt",
        clip_grad=None,
        scheduler=None,
        use_wandb=False,
        resume=False):

    history = {
        "train_loss": [],
        "train_kl": [],
        "train_cc": [],
        "train_nss": [],
        "val_loss": [],
        "val_kl": [],
        "val_cc": [],
        "val_nss": [],
    }

    best_nss = float("-inf")
    start_epoch = 1

    model.to(device)

    if resume:
        start_epoch, best_nss, history = load_checkpoint(
            checkpoint_path, model, optimizer, device
        )
        start_epoch += 1
        print(f"Resumed from epoch {start_epoch}")

    for epoch in range(start_epoch, epochs + 1):
        print(f"Epoch {epoch}/{epochs}")

        train_metrics = train_one_epoch(model, train_loader, optimizer, device, clip_grad)
        val_metrics = validate(model, valid_loader, device)

        if scheduler is not None:
            scheduler.step()

        history["train_loss"].append(train_metrics["loss"])
        history["train_kl"].append(train_metrics["kl"])
        history["train_cc"].append(train_metrics["cc"])
        history["train_nss"].append(train_metrics["nss"])

        history["val_loss"].append(val_metrics["loss"])
        history["val_kl"].append(val_metrics["kl"])
        history["val_cc"].append(val_metrics["cc"])
        history["val_nss"].append(val_metrics["nss"])

        if use_wandb:
            wandb.log({
                "epoch": epoch,
                "train_loss": train_metrics["loss"],
                "train_kl": train_metrics["kl"],
                "train_cc": train_metrics["cc"],
                "train_nss": train_metrics["nss"],
                "val_loss": val_metrics["loss"],
                "val_kl": val_metrics["kl"],
                "val_cc": val_metrics["cc"],
                "val_nss": val_metrics["nss"],
                "lr": optimizer.param_groups[0]["lr"]
            })

        if val_metrics["nss"] > best_nss:
            best_nss = val_metrics["nss"]
            torch.save(model.state_dict(), save_path)
            print(f"Saved best model to {save_path} with val_nss={best_nss:.4f}")

        save_checkpoint(
            checkpoint_path,
            model,
            optimizer,
            epoch,
            best_nss,
            history
        )
        print(f"Saved checkpoint to {checkpoint_path}")

    return history


Mounted at /content/drive


In [ ]:
import wandb

total_epochs = 30
batch_size = 28

wandb.init(
    project="saliency-prediction",
    config={
        "epochs": total_epochs,
        "lr": 1e-4,
        "batch_size": batch_size,
        "model": "SaliencyResNetTransPretrain",
        "loss": "kl + 0.5*(1-cc) - 0.3*nss"
    }
)

use_wandb = True

from google.colab import drive
drive.mount("/content/drive")

checkpoint_dir = "/content/drive/MyDrive/saliency_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

best_path = f"{checkpoint_dir}/best_resNet_trans_pretrain.pt"
checkpoint_path = f"{checkpoint_dir}/checkpoint_resNet_trans_pretrain.pt"

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SaliencyResNetTransPretrained(pretrained=True, freeze_stem=False, freeze_layer1=False).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

if use_wandb:
    wandb.watch(model, log="all", log_freq=100)

warmup_epochs = 3

warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=warmup_epochs
)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=total_epochs - warmup_epochs,
    eta_min=1e-6
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warmup_epochs]
)

history = fit(
    model,
    train_loader,
    val_loader,
    optimizer,
    device=device,
    epochs=total_epochs,
    save_path=best_path,
    checkpoint_path=checkpoint_path,
    scheduler=scheduler,
    use_wandb=use_wandb,
    resume=False
)

if use_wandb:
    wandb.finish()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: synnove2003 (synnove2003-ntnu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 152MB/s]


Epoch 1/30


Saved best model to /content/drive/MyDrive/saliency_checkpoints/best_resNet_trans_pretrain_NLL.pt with val_nss=1.6681
Saved checkpoint to /content/drive/MyDrive/saliency_checkpoints/checkpoint_resNet_trans_pretrain_NLL.pt
Epoch 2/30


...

Saved checkpoint to /content/drive/MyDrive/saliency_checkpoints/checkpoint_resNet_trans_pretrain_NLL.pt
Epoch 13/30


Saved checkpoint to /content/drive/MyDrive/saliency_checkpoints/checkpoint_resNet_trans_pretrain_NLL.pt
Epoch 14/30


Saved checkpoint to /content/drive/MyDrive/saliency_checkpoints/checkpoint_resNet_trans_pretrain_NLL.pt
Epoch 15/30


Saved checkpoint to /content/drive/MyDrive/saliency_checkpoints/checkpoint_resNet_trans_pretrain_NLL.pt
Epoch 16/30


Valid:  83%|████████▎ | 75/90 [00:53<00:10,  1.42it/s, loss=0.983]

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

best_model_path = "/content/drive/MyDrive/saliency_checkpoints/best_resNet_trans_pretrain.pt"

model.load_state_dict(torch.load(best_model_path, map_location=device))
model.to(device)
model.eval()


def auc_judd_single(pred, fixation, eps=1e-8):
    pred = pred.detach().cpu().numpy().squeeze()
    fixation = fixation.detach().cpu().numpy().squeeze() > 0

    if fixation.sum() == 0:
        return np.nan

    pred = (pred - pred.min()) / (pred.max() - pred.min() + eps)
    thresholds = np.sort(pred[fixation])[::-1]

    tp = []
    fp = []

    num_fix = fixation.sum()
    num_nonfix = fixation.size - num_fix

    for thresh in thresholds:
        above = pred >= thresh
        tp.append((above & fixation).sum() / num_fix)
        fp.append((above & ~fixation).sum() / num_nonfix)

    tp = np.array([0] + tp + [1])
    fp = np.array([0] + fp + [1])

    return np.trapz(tp, fp)


def auc_judd_batch(pred, fixation):
    scores = []

    for i in range(pred.shape[0]):
        score = auc_judd_single(pred[i], fixation[i])
        if not np.isnan(score):
            scores.append(score)

    return float(np.mean(scores)) if len(scores) > 0 else np.nan


def evaluate_saliency_model(model, dataloader, device):
    total_nss = 0.0
    total_cc = 0.0
    total_kl = 0.0
    total_auc = 0.0
    n = 0

    with torch.no_grad():
        for x, saliency, fixation in tqdm(dataloader, desc="Evaluating"):

            x = x.to(device)
            saliency = saliency.to(device)
            fixation = fixation.to(device)

            preds = model(x)
            pred_pos = F.softplus(preds)

            bs = x.shape[0]

            total_nss += (-nss_loss(pred_pos, fixation).item()) * bs
            total_cc += batch_cc(pred_pos, saliency) * bs
            total_kl += kl_spatial_loss(preds, saliency).item() * bs
            total_auc += auc_judd_batch(pred_pos, fixation) * bs

            n += bs

    return {
        "NSS": total_nss / n,
        "CC": total_cc / n,
        "KL": total_kl / n,
        "AUC-Judd": total_auc / n,
    }


salicon_test_results = evaluate_saliency_model(model, test_loader, device)

for k, v in salicon_test_results.items():
    print(f"{k}: {v:.4f}")

Evaluating:   0%|          | 0/90 [00:00<?, ?it/s]/tmp/ipykernel_827/2883692207.py:64: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(tp, fp)
Evaluating: 100%|██████████| 90/90 [11:55<00:00,  7.95s/it]


===== SALICON HELD-OUT TEST RESULTS =====
NSS: 1.8962
CC: 0.8637
KL: 0.2771
AUC-Judd: 0.8603
